# Phase 3: Exploratory Data Analysis & Preprocessing

**Objective:** Clean the dataset, explore feature distributions, handle categorical variables, and prepare the final train/test splits for model training.

**Outputs:**
- `data/X_train_raw.csv`, `data/X_test_raw.csv` (Raw features for tree models)
- `data/X_train.npy`, `data/X_test.npy` (Scaled features for linear models)
- `data/y_train.npy`, `data/y_test.npy` (Labels)
- `models/tld_encoder.pkl` (Categorical encoder)
- `models/scaler.pkl` (StandardScaler fitted on train set)

---
## 3.1 Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

In [2]:
df = pd.read_csv('../data/PhiUSIIL_Phishing_URL_Dataset.csv')
print(df.shape)
df.head()

---
## 3.2 Data Cleaning

Check for missing values and exact URL duplicates.

In [3]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate URLs:", df['URL'].duplicated().sum())

Missing values per column:
Series([], dtype: int64)

Duplicate rows: 0
Duplicate URLs: 425


In [4]:
df = df.drop_duplicates(subset='URL').reset_index(drop=True)
print("Shape after dedup:", df.shape)

Shape after dedup: (235370, 55)


---
## 3.3 Exploratory Data Analysis (EDA)

### Class Distribution

In [5]:
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True) * 100)

sns.countplot(data=df, x='label')
plt.title('Class Distribution (1=Legitimate, 0=Phishing)')
plt.show()

### Key Feature Distributions

Visualizing the distribution of a few hand-picked features across the two classes.

In [6]:
key_features = ['URLLength', 'DomainLength', 'URLSimilarityIndex', 
                 'NoOfSubDomain', 'IsHTTPS', 'NoOfExternalRef']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    sns.boxplot(data=df, x='label', y=feat, ax=axes[i])
    axes[i].set_title(feat)

plt.tight_layout()
plt.show()

### Correlation Heatmap

Overview of correlations between all numeric features.

In [7]:
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(20, 16))
sns.heatmap(numeric_df.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Heatmap')
plt.show()

---
## 3.4 Feature Selection & Encoding

Drop raw text columns (`URL`, `Domain`, `Title`) as they cannot be fed directly into standard models. Then encode the categorical `TLD` feature.

In [8]:
# URL, Domain, Title are raw text/identifiers — not features for tree/linear models directly
# TLD is categorical — needs encoding, handled next
cols_to_drop = ['URL', 'Domain', 'Title']
 
df_model = df.drop(columns=cols_to_drop)
print(df_model.shape)
df_model.columns.tolist()

In [9]:
le = LabelEncoder()
df_model['TLD_encoded'] = le.fit_transform(df_model['TLD'])
df_model = df_model.drop(columns=['TLD'])

print("Number of unique TLDs:", len(le.classes_))
df_model[['TLD_encoded']].head()

In [10]:
import pickle
with open('../models/tld_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

---
## 3.5 Train / Test Split

Perform an 80/20 stratified split to preserve the class balance in both sets.

In [11]:
X = df_model.drop(columns=['label'])
y = df_model['label']

print("X shape:", X.shape)
print("y shape:", y.shape)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True))
print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

Train shape: (188296, 51)
Test shape: (47074, 51)

Train class balance:
label
1    0.572928
0    0.427072
Name: proportion, dtype: float64

Test class balance:
label
1    0.57293
0    0.42707
Name: proportion, dtype: float64


---
## 3.6 Scaling

Linear models and Neural Networks require feature scaling. Tree-based models (like XGBoost) do not. We will save both versions.

In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# save scaler for later use in deployment
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [14]:
np.save('../data/X_train.npy', X_train_scaled)
np.save('../data/X_test.npy', X_test_scaled)
np.save('../data/y_train.npy', y_train.values)
np.save('../data/y_test.npy', y_test.values)

# also save unscaled versions (tree models like XGBoost don't need scaling)
X_train.to_csv('../data/X_train_raw.csv', index=False)
X_test.to_csv('../data/X_test_raw.csv', index=False)

print("All splits saved to data/")

All splits saved to data/


---
## Summary

### What was accomplished:
- ✅ Dataset cleaned (425 duplicates removed)
- ✅ Categorical `TLD` feature successfully encoded (`tld_encoder.pkl` saved)
- ✅ Extraneous text columns (`URL`, `Domain`, `Title`) dropped
- ✅ 80/20 stratified split created
- ✅ Output arrays saved (`X_train.npy`, `X_train_raw.csv`, etc.)

### Next step: Phase 4 — Leakage Investigation & Model Training